# KG Construction — Colab extraction

This notebook performs the longest GPU stage: it downloads the project and Re-DocRED, starts Ollama with `qwen2.5:3b`, and saves model responses to a resumable JSONL file on Google Drive.

If Colab disconnects, rerun the setup cells and the main extraction cell. Documents that were already processed successfully will be skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Settings

Keep the default values for the complete frozen TEST run. `BENCHMARK_DOCS = 20` processes 20 documents first; the full run then continues the same cache without repeating requests.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/DKR-KG-Construction/KG-construction.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/KG-construction')
DRIVE_ROOT = Path('/content/drive/MyDrive/DKR_KG_Construction')

MODEL = 'qwen2.5:3b'
SPLIT = 'test'          # train | dev | test
BENCHMARK_DOCS = 20     # set to 0 to skip the separate benchmark
RUN_FULL_EXTRACTION = True

RAW_PATH = DRIVE_ROOT / 'outputs' / 'raw' / f'{SPLIT}.jsonl'
OLLAMA_MODELS = DRIVE_ROOT / 'ollama_models'
RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
OLLAMA_MODELS.mkdir(parents=True, exist_ok=True)
print('Raw cache:', RAW_PATH)
print('Ollama models:', OLLAMA_MODELS)

## 2. GPU, repository, and dependencies

In [ ]:
import os
import shutil
import subprocess
import sys

def run(command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

run(['nvidia-smi'])

if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)])
else:
    run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH])
    run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])

run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements.lock')])
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
print('Repository commit:')
run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'])

## 3. Download and verify Re-DocRED

The script verifies the pinned SHA-256 checksums. After a Colab restart, the dataset is downloaded again, while the LLM cache remains on Drive.

In [ ]:
run(['bash', 'run.sh', 'data'], cwd=REPO_DIR)

## 4. Install and start Ollama

The model is stored on Google Drive, so it normally does not need to be downloaded again in a new session. The first run may take some time to load the model into VRAM.

In [ ]:
import time
import urllib.request

if shutil.which('ollama') is None:
    if shutil.which('zstd') is None:
        run(['apt-get', 'update', '-qq'])
        run(['apt-get', 'install', '-y', 'zstd'])
    ollama_installer = Path('/tmp/install_ollama.sh')
    urllib.request.urlretrieve('https://ollama.com/install.sh', ollama_installer)
    run(['sh', str(ollama_installer)])

os.environ['OLLAMA_MODELS'] = str(OLLAMA_MODELS)
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'

def ollama_ready():
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2).read()
        return True
    except Exception:
        return False

if not ollama_ready():
    ollama_log = open('/content/ollama.log', 'ab', buffering=0)
    ollama_process = subprocess.Popen(
        ['ollama', 'serve'],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    for _ in range(60):
        if ollama_ready():
            break
        time.sleep(1)
    else:
        raise RuntimeError('Ollama did not start. Inspect /content/ollama.log')

run(['ollama', 'pull', MODEL], env=os.environ.copy())
run(['ollama', 'list'], env=os.environ.copy())

## 5. Optional benchmark on the first documents

The benchmark writes to the same JSONL file, so its results are reused by the full run. If the cache already exists, this stage simply resumes it.

In [ ]:
import time

def extraction_command(n_docs):
    return [
        sys.executable, '-m', 'kgrepair',
        '--config', str(REPO_DIR / 'config.yaml'),
        'extract', '--split', SPLIT, '--n', str(n_docs), '--out', str(RAW_PATH),
    ]

if BENCHMARK_DOCS:
    started = time.perf_counter()
    run(extraction_command(BENCHMARK_DOCS), cwd=REPO_DIR, env=os.environ.copy())
    elapsed = time.perf_counter() - started
    print(f'Benchmark wall time: {elapsed / 60:.1f} min')
    print(f'Rough 500-document estimate: {elapsed * 500 / BENCHMARK_DOCS / 3600:.1f} h')

## 6. Full resumable extraction

This cell may run for many hours. Every completed record is appended to `RAW_PATH` immediately. After a session disconnect, restart the notebook and run this cell again.

In [ ]:
if RUN_FULL_EXTRACTION:
    run(extraction_command('all'), cwd=REPO_DIR, env=os.environ.copy())
else:
    print('RUN_FULL_EXTRACTION=False: full extraction skipped')

## 7. Validate the cache

TEST should contain 500 unique documents. Unparseable responses remain part of the experiment and produce an empty graph during evaluation.

In [ ]:
sys.path.insert(0, str(REPO_DIR / 'src'))
from kgrepair.extract import load_raw

raw = load_raw(RAW_PATH)
normal = sum(r['parsed'] is not None and r.get('parse_error') is None for r in raw.values())
partial = sum(r.get('parse_error') == 'partial_json_recovered' for r in raw.values())
failed = sum(r['parsed'] is None for r in raw.values())
print({'unique_documents': len(raw), 'normal': normal, 'partial': partial, 'failed': failed})
print('Saved to:', RAW_PATH)

Continue with `04_colab_evaluate_annotate.ipynb`. That stage does not require a GPU.